# **Causal scrubbing, иллюзии и MDL probing: практика**

Тетрадь к модулю «Оценка интерпретируемости и научная строгость». В ней собраны toy-прогоны уроков модуля, и числа в шагах уроков взяты из ее вывода:

* урок «Faithfulness: верна ли интерпретация»: causal scrubbing трех гипотез о модели с известным устройством и доля восстановленной потери $R(H)$;
* урок «Иллюзии интерпретируемости»: направление, которое внутри корпуса выглядит детектором темы (как у Bolukbasi et al.), и заместитель головы внимания, верный на своем распределении и неверный вне его (как у Friedman et al.);
* урок «Подводные камни probing»: точность и длина онлайн-кода (Voita и Titov) линейного зонда и MLP-зонда на настоящей и контрольной задаче (Hewitt и Liang).

Хватит CPU: на трех потоках процессора наш прогон всей тетради занял 7.5 минуты, почти все — обучение зондов в разделе 3. Сиды стоят в каждой ячейке, поэтому числа повторяются при повторном запуске. Задания пронумерованы так же, как в шагах урока «Causal scrubbing, иллюзии и MDL probing: Практика».

## **Подготовка окружения**
Нужны только torch и numpy: в Colab они уже стоят. Все считается на CPU, чтобы числа совпали с шагами урока и на машине с GPU.

In [ ]:
import math, json, time
import numpy as np
import torch
import torch.nn.functional as F

R = {}  # all numbers of the notebook; saved to toys_32_results.json at the end
T0 = time.time()
print("torch", torch.__version__)

## **1. Causal scrubbing на модели с известным устройством**

Вход — четыре признака $x_1, x_2, x_3, x_4 \in \{-1, +1\}$; метка $y = 1$, когда знаки $x_1$ и $x_2$ совпадают. Признаки $x_1, x_2, x_4$ независимы и равновероятны, $x_3$ совпадает с произведением $x_1 x_2$ с вероятностью 0.8. Модель не обучаем, ее логит задан формулой $z = 3\,x_1 x_2 + 1.5\,x_3$; поведение — средняя бинарная кросс-энтропия по метке $y$ в натах.

Гипотеза объявляет часть признаков неважными; их значения берем с другого случайного примера той же выборки (одна случайная перестановка строк на гипотезу), важные признаки остаются своими. Доля восстановленной потери по Chan et al.:

$$R(H) = \frac{L_{\mathrm{base}} - L_{\mathrm{scrub}}}{L_{\mathrm{base}} - L_{\mathrm{orig}}},$$

где $L_{\mathrm{base}}$ — потеря исходной модели против меток, переставленных случайной перестановкой. $N = 40\,000$ примеров, сид 0.

In [ ]:
torch.manual_seed(0)
N = 40000
x = torch.randint(0, 2, (N, 4)).float() * 2 - 1
agree = torch.rand(N) < 0.8
x[:, 2] = torch.where(agree, x[:, 0] * x[:, 1], -x[:, 0] * x[:, 1])
y = (x[:, 0] * x[:, 1] > 0).float()

logit = lambda x: 3 * x[:, 0] * x[:, 1] + 1.5 * x[:, 2]
bce = lambda z, y: F.binary_cross_entropy_with_logits(z, y).item()
accuracy = lambda z, y: ((z > 0).float() == y).float().mean().item()

L_orig = bce(logit(x), y)
L_base = bce(logit(x), y[torch.randperm(N)])
R["scrub"] = {"L_orig": L_orig, "acc_orig": accuracy(logit(x), y), "L_base": L_base,
              "share_x3_eq_x1x2": (x[:, 2] == x[:, 0] * x[:, 1]).float().mean().item(), "share_y1": y.mean().item()}
print(f"L_orig = {L_orig:.3f}, accuracy of the original model {R['scrub']['acc_orig']:.3f}, L_base = {L_base:.3f}")
print(f"share of x3 = x1*x2: {R['scrub']['share_x3_eq_x1x2']:.3f}, share of y = 1: {R['scrub']['share_y1']:.3f}")

HYPOTHESES = {"A": [0, 1], "B": [0, 2], "C": [0, 1, 2]}  # indices of the important features
print(f"{'hypothesis':<11} {'L_scrub':>8} {'accuracy':>9} {'R(H)':>6}")
for name, important in HYPOTHESES.items():
    xs = x[torch.randperm(N)].clone()
    xs[:, important] = x[:, important]
    z = logit(xs)
    L_scrub = bce(z, y)
    R["scrub"][name] = {"L_scrub": L_scrub, "acc": accuracy(z, y), "R": (L_base - L_scrub) / (L_base - L_orig)}
    print(f"{name:<11} {L_scrub:>8.3f} {R['scrub'][name]['acc']:>9.3f} {R['scrub'][name]['R']:>6.3f}")

**Задание 1. Какую долю потери $R(H)$ восстанавливает гипотеза B, которая объявляет неважными $x_2$ и $x_4$? Ответ дайте с тремя знаками после точки.**

Гипотеза B опускает точность почти до уровня угадывания, но $R$ у нее не 0: $x_3$ совпадает с $x_1 x_2$ в 80% примеров и частично снижает потерю.

## **2. Две иллюзии интерпретируемости**

**Часть 1, как у Bolukbasi et al.** Два корпуса по 10 000 точек в $\mathbb R^{16}$: центр корпуса A — $+4e_0$, корпуса B — $-4e_0$. В каждом корпусе 5 тем по 2000 точек: центр темы отстоит от центра корпуса на 2.5 в случайном направлении, шум со стандартным отклонением 0.5. Берем 100 случайных единичных направлений $v$ и у каждого 10 точек корпуса с наибольшей проекцией на $v$. Чистота набора — доля точек самой частой темы среди 10. Для сравнения — средняя чистота 10 случайных точек корпуса A по 1000 выборкам. Сид 0.

In [ ]:
torch.manual_seed(0)
D = 16

def corpus(sign):
    center = torch.zeros(D); center[0] = 4.0 * sign
    points, topics = [], []
    for k in range(5):
        u = torch.randn(D); u /= u.norm()
        points.append(center + 2.5 * u + 0.5 * torch.randn(2000, D)); topics.append(torch.full((2000,), k + 1))
    return torch.cat(points), torch.cat(topics)

XA, TA = corpus(+1)
XB, TB = corpus(-1)
V = torch.randn(100, D); V /= V.norm(dim=1, keepdim=True)

def purity(topics, idx):
    counts = torch.bincount(topics[idx], minlength=6)
    return counts.max().item() / len(idx), counts.argmax().item(), counts.max().item()

pA = [purity(TA, (XA @ v).topk(10).indices) for v in V]
pB = [purity(TB, (XB @ v).topk(10).indices) for v in V]
p_rand = [purity(TA, torch.randperm(10000)[:10])[0] for _ in range(1000)]
R["illusion1"] = {"mean_purity_A": float(np.mean([p[0] for p in pA])), "pure_A": sum(p[0] == 1 for p in pA),
                  "mean_purity_B": float(np.mean([p[0] for p in pB])), "pure_B": sum(p[0] == 1 for p in pB),
                  "mean_purity_random_A": float(np.mean(p_rand)), "first_direction_A": pA[0][1:], "first_direction_B": pB[0][1:]}
print(f"top 10 along v, corpus A: mean purity {R['illusion1']['mean_purity_A']:.3f}, directions with purity 1.0: {R['illusion1']['pure_A']} of 100")
print(f"top 10 along v, corpus B: mean purity {R['illusion1']['mean_purity_B']:.3f}, directions with purity 1.0: {R['illusion1']['pure_B']} of 100")
print(f"10 random points of corpus A, 1000 draws: mean purity {R['illusion1']['mean_purity_random_A']:.3f}")
print(f"first direction: corpus A — {pA[0][2]} of 10 from topic {pA[0][1]}, corpus B — {pB[0][2]} of 10 from topic {pB[0][1]}")

**Задание 2. Какова средняя чистота верхних 10 точек по 100 случайным направлениям в корпусе A? Ответ дайте с тремя знаками после точки.**

Внутри каждого корпуса направление выглядит детектором одной темы, хотя темы здесь — кластеры, и вдоль $v$ ни одна тема по всему пространству не нарастает.

**Часть 2, как у Friedman et al.** Запрос $q$ и 16 ключей $k_i$ в $\mathbb R^8$, скор $s_i = q^\top k_i$, картина внимания — $\mathrm{softmax}(s)$, предсказание — ключ с наибольшим скором. На ID-данных (из распределения, где подгоняют заместителя) стандартное отклонение координат 0–2 равно 1, координат 3–7 — 0.05; на OOD-данных (вне этого распределения) у всех координат 1. Заместитель проецирует $q$ и $k_i$ на первые 3 правых сингулярных вектора матрицы, в которой строки — запросы и ключи 1000 ID-последовательностей. Оценка — по 2000 последовательностей каждого вида: доля совпадающих предсказаний и средняя дивергенция Йенсена–Шеннона (JSD, в натах) картин внимания модели и заместителя. Сид 0 продолжает генератор части 1.

In [ ]:
def sequences(n, in_distribution):
    sd = torch.ones(8)
    if in_distribution: sd[3:] = 0.05
    return torch.randn(n, 17, 8) * sd  # position 0 is the query, 1..16 are the keys

S = sequences(1000, True).reshape(-1, 8)
_, sing, Vt = torch.linalg.svd(S, full_matrices=False)
proj = Vt[:3].T @ Vt[:3]

def evaluate(Z):
    q, k = Z[:, 0], Z[:, 1:]
    s = torch.einsum("nd,nkd->nk", q, k); s_sur = torch.einsum("nd,nkd->nk", q @ proj, k @ proj)
    a, b = s.softmax(1), s_sur.softmax(1); m = (a + b) / 2
    kl = lambda p, r: (p * (p / r).log()).sum(1)
    return (s.argmax(1) == s_sur.argmax(1)).float().mean().item(), (0.5 * kl(a, m) + 0.5 * kl(b, m)).mean().item()

R["illusion2"] = {"singular_values": sing.tolist(), "ID": evaluate(sequences(2000, True)), "OOD": evaluate(sequences(2000, False))}
print("singular values of the ID matrix:", [round(v, 1) for v in R["illusion2"]["singular_values"]])
for kind in ["ID", "OOD"]:
    agree, jsd = R["illusion2"][kind]
    print(f"{kind:>3}: share of matching predictions {agree:.3f}, mean JSD {jsd:.3g}")

**Задание 3. Какая доля предсказаний заместителя совпадает с предсказаниями модели на OOD-данных? Ответ дайте с тремя знаками после точки.**

SVD считается по данным, поэтому заместитель описывает модель там, где его подгоняли: отброшенные направления почти не меняются на ID-данных, а на OOD-данных на них приходится 5 из 8 координат с одинаковым разбросом.

## **3. Точность и длина онлайн-кода на настоящей и контрольной задаче**

**Данные.** 600 типов слов, активации размерности $d = 48$, 20 000 вхождений, типы вхождений равновероятны. У каждого типа своя случайная сигнатура $2.0\cdot\mathcal N(0, I)$; свойство типа (0 или 1) записано слабо, сдвигом вдоль одного единичного направления $w_{\mathrm{prop}}$:

$$h = \mathrm{sig}[\text{тип}] + 0.3\cdot\mathcal N(0, I) + 1.6\,(2y - 1)\,w_{\mathrm{prop}}.$$

**Задачи.** Настоящая — метка свойства типа $y$; контрольная — каждому типу случайно приписана метка 0 или 1, как у Hewitt и Liang.

**Зонды.** Логистическая регрессия и MLP с двумя скрытыми слоями по 256 нейронов. Оба обучаются Adam, lr $10^{-3}$, 1500 шагов на всей обучающей части; веса выбираются по наименьшей потере на 10% обучающей части, отложенных для остановки, как правило остановки по dev-выборке у Voita и Titov.

**Онлайн-код.** Блоки по долям 0.1, 0.2, 0.4, 0.8, 1.6, 3.2, 6.25, 12.5, 25, 50 и 100% данных, как у Voita и Titov. Первый блок передается равномерным кодом, 1 бит на метку; каждый следующий блок кодирует зонд, обученный на всех предыдущих вхождениях, и стоит его суммарную кросс-энтропию на блоке в битах. Равномерный код всех меток — 20 000 бит, сжатие — 20 000, деленное на длину онлайн-кода. Точность — у зонда, обученного на первой половине вхождений, на второй половине. Сиды 0–4: сид задает и данные, и инициализацию зондов. Около 90 секунд на сид на трех потоках процессора.

In [ ]:
N_TYPES, DIM, N_OCC, HIDDEN, STEPS = 600, 48, 20000, 256, 1500
FRACTIONS = [0.1, 0.2, 0.4, 0.8, 1.6, 3.2, 6.25, 12.5, 25, 50, 100]

def make_data(seed):
    g = torch.Generator().manual_seed(seed)
    sig = 2.0 * torch.randn(N_TYPES, DIM, generator=g)
    w_prop = torch.randn(DIM, generator=g); w_prop /= w_prop.norm()
    prop = torch.randint(0, 2, (N_TYPES,), generator=g)
    control = torch.randint(0, 2, (N_TYPES,), generator=g)
    token = torch.randint(0, N_TYPES, (N_OCC,), generator=g)
    H = sig[token] + 0.3 * torch.randn(N_OCC, DIM, generator=g) + 1.6 * (2 * prop[token] - 1).float()[:, None] * w_prop
    return H, prop[token], control[token]

def make_probe(kind):
    if kind == "linear": return torch.nn.Linear(DIM, 2)
    return torch.nn.Sequential(torch.nn.Linear(DIM, HIDDEN), torch.nn.ReLU(), torch.nn.Linear(HIDDEN, HIDDEN), torch.nn.ReLU(), torch.nn.Linear(HIDDEN, 2))

def train(kind, X, y):
    model = make_probe(kind); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    n_stop = max(1, len(X) // 10)
    X_stop, y_stop, X_fit, y_fit = X[:n_stop], y[:n_stop], X[n_stop:], y[n_stop:]
    best, best_state = float("inf"), None
    for _ in range(STEPS):
        opt.zero_grad(); F.cross_entropy(model(X_fit), y_fit).backward(); opt.step()
        with torch.no_grad(): stop_loss = F.cross_entropy(model(X_stop), y_stop).item()
        if stop_loss < best: best, best_state = stop_loss, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

def online_code(kind, X, y):
    edges = [round(N_OCC * f / 100) for f in FRACTIONS]
    bits, half_accuracy = float(edges[0]), None  # first block: uniform code, log2(2) = 1 bit per label
    for start, end in zip(edges[:-1], edges[1:]):
        torch.manual_seed(start)
        model = train(kind, X[:start], y[:start])
        with torch.no_grad():
            bits += F.cross_entropy(model(X[start:end]), y[start:end], reduction="sum").item() / math.log(2)
            if start == N_OCC // 2: half_accuracy = (model(X[start:]).argmax(1) == y[start:]).float().mean().item()
    return half_accuracy, bits

R["mdl"] = {}
for seed in range(5):
    H, y_real, y_control = make_data(seed)
    for kind in ["linear", "mlp"]:
        for task, labels in [("real", y_real), ("control", y_control)]:
            acc, bits = online_code(kind, H, labels)
            R["mdl"].setdefault(f"{kind}/{task}", []).append({"seed": seed, "accuracy": acc, "bits": bits})
    print(f"seed {seed} done, {round(time.time() - T0)} s")

print(f"{'probe':<7} {'task':<8} {'accuracy':>9} {'online code, bits':>18} {'sd over seeds':>14} {'compression':>12}")
for key, rows in R["mdl"].items():
    kind, task = key.split("/")
    acc = np.mean([r["accuracy"] for r in rows]); bits = np.array([r["bits"] for r in rows])
    print(f"{kind:<7} {task:<8} {acc:>9.3f} {bits.mean():>18.0f} {bits.std(ddof=1):>14.0f} {N_OCC / bits.mean():>12.2f}")

**Задание 4. Во сколько раз онлайн-код MLP-зонда на контрольной задаче длиннее, чем на настоящей, по средним длинам кода по пяти сидам? Ответ дайте с двумя знаками после точки.**

По точности MLP-зонд настоящую задачу от контрольной не отличает: при 20 000 вхождениях 600 типов каждый тип встречается в обучающей половине, и MLP достаточно узнать тип по сигнатуре и вспомнить его метку. Настоящую метку можно предсказать по направлению $w_{\mathrm{prop}}$ для типа, которого зонд еще не видел, поэтому в ранних блоках она кодируется дешево; контрольную метку нового типа угадать нельзя.

In [ ]:
mlp_real = np.mean([r["bits"] for r in R["mdl"]["mlp/real"]]); mlp_control = np.mean([r["bits"] for r in R["mdl"]["mlp/control"]])
print(f"MLP probe: control / real code length = {mlp_control:.0f} / {mlp_real:.0f} = {mlp_control / mlp_real:.2f}")
json.dump(R, open("toys_32_results.json", "w"))
print("saved toys_32_results.json; total time", round(time.time() - T0), "s")

## **Итог**

* Число $R(H)$ отличает неполную гипотезу от полной там, где точность после замен одинакова, и зависит от метрики поведения: уровень угадывания по точности не совпадает с нулем шкалы $R$.
* Направление, найденное по верхним примерам внутри одного корпуса, и заместитель, подогнанный на одном распределении, верны там, где их нашли, и не переносятся за его пределы.
* Одинаковая точность зонда на настоящей и контрольной задаче не значит одинаковую информацию в активациях: длина онлайн-кода их различает.